# Statistical Analysis and Portfolio Optimization

## Overview

This notebook performs statistical analysis on the collected financial data and implements the Markowitz Mean-Variance portfolio optimization algorithm. We analyze asset returns, compute risk metrics, and determine optimal portfolio weights that maximize returns for a given level of risk.

### Key Objectives:
- Load and validate financial time series data
- Compute statistical measures (mean returns, volatilities, correlations)
- Build the covariance matrix for risk assessment
- Formulate and solve the portfolio optimization problem
- Analyze optimal asset allocation

### Mathematical Foundation:
The portfolio optimization problem minimizes variance subject to return and budget constraints:

$$\min_{w} \frac{1}{2} w^T \Sigma w$$
$$\text{subject to:} \quad w^T \mu \geq r_{\text{target}}, \quad w^T 1 = 1, \quad w \geq 0$$

Where:
- $w$: Portfolio weights
- $\Sigma$: Covariance matrix
- $\mu$: Expected returns vector
- $r_{\text{target}}$: Target return

In [ ]:
# Import necessary libraries for analysis and optimization
import numpy as np        # Numerical computations and array operations
import pandas as pd       # Data manipulation and analysis
import cvxpy as cp        # Convex optimization library for portfolio optimization

In [ ]:
# Load summary data (for reference)
df = pd.read_csv("Data.csv")
print("Summary Statistics from Data Collection:")
print(df)

# Extract expected returns from summary data
returns = df.iloc[0, 1:]  # First row contains returns
print("\nExpected Returns:")
print(returns)

  Unnamed: 0      AAPL      MSFT      NVDA       JPM        GS       XOM  \
0     Return -0.328679 -0.878170 -0.064026 -0.442125 -0.367244  1.311107   
1        STD  0.240025  0.316209  0.337322  0.243065  0.338451  0.260398   

        JNJ      AMZN        PG       GLD  
0  0.618199 -0.341856  0.115720  0.268640  
1  0.158610  0.290512  0.207762  0.402045  


AAPL   -0.328679
MSFT    -0.87817
NVDA   -0.064026
JPM    -0.442125
GS     -0.367244
XOM     1.311107
JNJ     0.618199
AMZN   -0.341856
PG       0.11572
GLD      0.26864
Name: 0, dtype: object

## Covariance Matrix Calculation

The summary statistics provide point estimates but cannot generate a proper covariance matrix. For portfolio optimization, we need the covariance between assets, which requires time series data showing how returns co-move over time.

Using the full returns dataset from `returns_data.csv`, we can compute:
- **Covariance Matrix**: Measures joint variability between assets
- **Correlation Matrix**: Standardized measure of relationships
- **Annualized Statistics**: Convert daily metrics to annual scale

In [ ]:
# Load the complete returns time series data
returns_df = pd.read_csv("returns_data.csv", index_col=0)  # Set dates as index

# Compute the covariance matrix and annualize (252 trading days)
cov_matrix = returns_df.cov() * 252
print("Annualized Covariance Matrix (10x10):")
print(cov_matrix.round(6))  # Round for readability

# Compute annualized expected returns and volatilities
mean_returns = returns_df.mean() * 252  # Annualized mean returns
std_devs = returns_df.std() * (252 ** 0.5)  # Annualized standard deviations

print("\nAnnualized Mean Returns:")
print(mean_returns.round(4))
print("\nAnnualized Standard Deviations:")
print(std_devs.round(4))

Covariance Matrix (10x10):
          AAPL      MSFT      NVDA       JPM        GS       XOM       JNJ  \
AAPL  0.057640  0.003792  0.023985  0.019647  0.032419  0.001407  0.002535   
MSFT  0.003792  0.100920  0.029708  0.009711  0.016471 -0.017557 -0.007787   
NVDA  0.023985  0.029708  0.118240  0.019208  0.054922  0.001964  0.002023   
JPM   0.019647  0.009711  0.019208  0.059122  0.053599  0.001853 -0.012350   
GS    0.032419  0.016471  0.054922  0.053599  0.114723 -0.007834 -0.009963   
XOM   0.001407 -0.017557  0.001964  0.001853 -0.007834  0.067886  0.001579   
JNJ   0.002535 -0.007787  0.002023 -0.012350 -0.009963  0.001579  0.025746   
AMZN  0.013969  0.031704  0.013138  0.016124  0.015981 -0.016671 -0.013726   
PG    0.005194 -0.022162 -0.014055 -0.006527 -0.016676  0.008774  0.012643   
GLD  -0.007883  0.011335  0.027189  0.009343  0.016566  0.035384  0.007333   

          AMZN        PG       GLD  
AAPL  0.013969  0.005194 -0.007883  
MSFT  0.031704 -0.022162  0.011335  
NVD

## Statistical Analysis Summary

From the time series data, we have computed:
- **Covariance Matrix**: Essential for measuring portfolio risk
- **Expected Returns**: Annualized mean returns for each asset
- **Volatilities**: Annualized standard deviations

These statistics form the foundation for portfolio optimization, where we seek to minimize risk (variance) while achieving a target return level.

## Portfolio Optimization Setup

Now we implement the core optimization algorithm. Using CVXPY, we formulate the Markowitz portfolio optimization as a quadratic programming problem.

### Problem Formulation:
- **Decision Variables**: Portfolio weights $w_i$ for each asset
- **Objective**: Minimize portfolio variance $\frac{1}{2} w^T \Sigma w$
- **Constraints**: Budget, return target, and non-negativity

Defining the decision variable: 

In [ ]:
# Define decision variables: portfolio weights for 10 assets
n = 10  # Number of assets
w = cp.Variable(n)  # Weight vector w = [w1, w2, ..., w10]
print("Portfolio weights variable:", w)

Variable((10,), var1)

## Objective Function

The objective is to minimize portfolio variance (risk):

$$\min_{w} \quad \frac{1}{2} w^T \Sigma w$$

This quadratic form represents the portfolio's total risk, where $\Sigma$ is the covariance matrix capturing both individual asset volatilities and correlations between assets.

In [ ]:
# Define the objective function: minimize portfolio variance
obj = w.T @ cov_matrix @ w  # Quadratic form: w^T * Σ * w
obj_func = cp.Minimize(obj)
print("Objective function:", obj_func)

Minimize(Expression(CONVEX, NONNEGATIVE, ()))

## Constraints

The optimization is subject to three key constraints:

1. **Budget Constraint**: All weights must sum to 1 (fully invested)
   $$w_1 + w_2 + \dots + w_{10} = 1$$

2. **Return Constraint**: Portfolio must achieve minimum expected return
   $$\mu^T w \geq r_{\text{target}}$$

3. **Non-negativity**: No short selling allowed
   $$w_i \geq 0 \quad \forall i$$

Obtaining the data on max return to set an expected return value:

In [29]:
# Set target expected return
# First, check the maximum possible return for reference
max_return = mean_returns.max()
print(f"Maximum expected return: {max_return:.4f}")

# Set target return (15% annual return)
e_return = 0.15
print(f"Target expected return: {e_return:.2%}")

Maximum expected return: 1.3294
Target expected return: 15.00%


Setting the constraints to cvxpy

In [30]:
# Extract expected returns as numpy array
mu = mean_returns.values

# Define optimization constraints
constraints = [
    cp.sum(w) == 1,           # Budget constraint: weights sum to 1
    mu.T @ w >= e_return,     # Return constraint: achieve target return
    w >= 0                    # Non-negativity: no short selling
]

print("Constraints defined:")
print(f"1. Sum of weights = 1")
print(f"2. Portfolio return >= {e_return:.2%}")
print(f"3. All weights >= 0")

Constraints defined:
1. Sum of weights = 1
2. Portfolio return >= 15.00%
3. All weights >= 0


Defining the optimization problem and using the optimization algorithm from cvpxy

In [31]:
# Formulate the optimization problem
prob = cp.Problem(obj_func, constraints)

# Solve the quadratic programming problem
result = prob.solve()

print(f"Optimization Status: {prob.status}")
print(f"Optimal Portfolio Variance: {prob.value:.6f}")

Optimization Status: optimal
Optimal Portfolio Variance: 0.007120


In [32]:
# Display optimization results
print(f"Problem Status: {prob.status}")
print(f"Optimal Value (Minimum Variance): {prob.value:.6f}")
print(f"Optimal Weights: {w.value}")
print(f"Sum of Weights: {np.sum(w.value):.6f}")

# Display weights by asset
tickers = ['AAPL', 'MSFT', 'NVDA', 'JPM', 'GS', 'XOM', 'JNJ', 'AMZN', 'PG', 'GLD']
print("\nOptimal Portfolio Allocation:")
for ticker, weight in zip(tickers, w.value):
    if weight > 1e-6:  # Only show significant weights
        print(f"{ticker}: {weight:.4f} ({weight*100:.2f}%)")

Problem Status: optimal
Optimal Value (Minimum Variance): 0.007120
Optimal Weights: [-1.21511162e-23  9.73004782e-02 -2.28300905e-23  1.35441984e-01
  2.89200199e-02  1.31889923e-01  3.71232244e-01  1.10042136e-01
  1.25173213e-01  6.23580601e-23]
Sum of Weights: 1.000000

Optimal Portfolio Allocation:
MSFT: 0.0973 (9.73%)
JPM: 0.1354 (13.54%)
GS: 0.0289 (2.89%)
XOM: 0.1319 (13.19%)
JNJ: 0.3712 (37.12%)
AMZN: 0.1100 (11.00%)
PG: 0.1252 (12.52%)


## Results Interpretation

The optimization successfully found the minimum-variance portfolio that achieves at least 15% expected return. Note that very small negative weights (near machine precision, ~1e-23) are numerical artifacts and should be treated as zero.

### Key Insights:
- The algorithm allocates capital to achieve the target return with minimal risk
- Assets with negative weights are effectively excluded from the portfolio
- The solution represents a point on the efficient frontier

### Next Steps:
- Analyze the efficient frontier by varying the target return
- Visualize portfolio performance and risk metrics
- Consider additional constraints (transaction costs, sector limits, etc.)